# Explore MCF skyddsrum API

In [151]:
import httpx

response = httpx.get(
    "https://inspire.mcf.se/skyddsrum/ogc/features/v1",
    params = {"limit": 1}
)
response.json()


{'title': 'OGC API - Features: Skyddsrum / Civil defence shelters',
 'description': 'Skyddsrum',
 'links': [{'href': 'https://inspire.mcf.se/skyddsrum/ogc/features/v1/?f=application%2Fjson',
   'rel': 'self',
   'type': 'application/json',
   'title': 'This document'},
  {'href': 'https://inspire.mcf.se/skyddsrum/ogc/features/v1/?f=application%2Fyaml',
   'rel': 'alternate',
   'type': 'application/yaml',
   'title': 'This document as application/yaml'},
  {'href': 'https://inspire.mcf.se/skyddsrum/ogc/features/v1/?f=text%2Fhtml',
   'rel': 'alternate',
   'type': 'text/html',
   'title': 'This document as text/html'},
  {'href': 'https://inspire.mcf.se/skyddsrum/ogc/features/v1/openapi?f=application%2Fvnd.oai.openapi%2Bjson%3Bversion%3D3.0',
   'rel': 'service-desc',
   'type': 'application/vnd.oai.openapi+json;version=3.0',
   'title': 'API definition for this endpoint as application/vnd.oai.openapi+json;version=3.0'},
  {'href': 'https://inspire.mcf.se/skyddsrum/ogc/features/v1/open

In [152]:
response.headers.keys()

dict_keys(['date', 'server', 'x-content-type-options', 'x-frame-options', 'content-security-policy', 'api-version', 'link', 'content-type', 'access-control-allow-origin', 'access-control-allow-headers', 'access-control-allow-methods', 'keep-alive', 'connection', 'transfer-encoding'])

In [153]:
response.status_code

200

## Explore data for coordinate areas

In [170]:
re = httpx.get(
    "https://inspire.mcf.se/geoserver/skyddsrum/ogc/features/v1/collections/US_civilProtectionSite",
    params = {"f": "json"}
)
re.json()
data = re.json()

In [155]:
type(data)

dict

In [156]:
data.keys()

dict_keys(['id', 'title', 'description', 'extent', 'links', 'crs', 'storageCrs'])

In [157]:
data["extent"]["spatial"]["bbox"]

[[9.74245055, 55.17669713, 24.85581961, 67.9515059]]

### Claculate lon and lat:

format in API "[9.74245055, 55.17669713, 24.85581961, 67.9515059]"
- min lon, min lat, max lon, max lat
- center_lon = (min_lon + max_lon) / 2
- center_lat = (min_lat + max_lat) / 2
- center_point = (center_log, center_lat)

### Get lon and lat:
- min_lon = center_lon - d (distance)
- max_lon = center_lon + d
- min_lat = center_lat - d
- max_lat = center_lat + d

In [158]:
# calculate
import math

def calculate_bbox(lon: float, lat: float, d_meter = 5000):
    d_lat = d_meter/111320
    d_lon = d_meter/(111320 * math.cos(math.radians(lat)))
    return [lon - d_lon, lat - d_lat, lon + d_lon, lat + d_lat]


In [159]:
lon_c=18.0123
lat_c=59.3011
bbox = calculate_bbox(lon_c, lat_c)

In [160]:
bbox

[17.924321086698104, 59.25618444125045, 18.100278913301896, 59.34601555874955]

## Deeper exploration in data

In [175]:
bbox_get = httpx.get(
        "https://inspire.mcf.se/geoserver/skyddsrum/ogc/features/v1/collections/US_civilProtectionSite/items",
        params = {
        "f": "json",
        "bbox": ",".join(map(str, bbox)),
        "limit": 100
        }
    )
bbox_get.json()
bbox_data = bbox_get.json()

In [162]:
bbox_data.keys()

dict_keys(['type', 'features', 'totalFeatures', 'numberMatched', 'numberReturned', 'timeStamp', 'links', 'crs', 'bbox'])

In [163]:
bbox_data["features"]

[{'type': 'Feature',
  'id': 'US_civilProtectionSite.11093',
  'geometry': {'type': 'MultiPoint',
   'coordinates': [[17.98231774, 59.30612537]]},
  'geometry_name': 'geom',
  'properties': {'additionallocationdescription': 'Provröret 15',
   'beginlifespanversion': '20261126',
   'endlifespanversion': '',
   'inspireid': 'ff2f73db-53d0-478a-ad88-a51038251e1d',
   'name': '115450-4',
   'numberofoccupants': 180,
   'pointofcontact': 'The Swedish Civil Defence and Resilience Agency (MCF)',
   'relatedparty': 'The Swedish Civil Defence and Resilience Agency (MCF)',
   'servicelevel': 'STA',
   'servicetype': 'civilProtectionSite',
   'typeofoccupant': 'Civilbefolkning',
   'status': 3,
   'statustype': 'Skyddsrum',
   'variantid': 1,
   'variantnamn': 'Normal',
   'id': 11093},
  'bbox': [17.98231774, 59.30612537, 17.98231774, 59.30612537]},
 {'type': 'Feature',
  'id': 'US_civilProtectionSite.11094',
  'geometry': {'type': 'MultiPoint',
   'coordinates': [[17.98226016, 59.30627046]]},
 

## Function for closest skyddsrum

In [ ]:
# get skyddsrum in a box [min_lon, min_lat, max_lon, max_lat]

def get_data(bbox):
    bbox_get = httpx.get(
        "https://inspire.mcf.se/geoserver/skyddsrum/ogc/features/v1/collections/US_civilProtectionSite/items", # use /items to get the data
        params = {
        "f": "json",
        "bbox": ",".join(map(str, bbox)), # [a,b,c,d] -> "a,b,c,d" (to give a str til the API)
        }
    )
    data = bbox_get.json()

    # only include skyddsrum 
    data["features"] = [
        f for f in data.get("features", [])
        if f["properties"].get("statustype") == "Skyddsrum"
    ]
    return data

In [188]:
# haversine: calculate the great-circle distance between two points on a sphere from their latitudes and longitudes 
# with regard to the globe beeing slimmer in towards the poles

# help from LLM to understand haversine and how to calculate it
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

In [ ]:
def closest_skyddsrum(lat_c, lon_c):

    bbox = calculate_bbox(lon=lon_c, lat=lat_c)
    data = get_data(bbox)

    # get features from API, empty list if empty
    features = data.get("features", [])

    
    result = []
    # read lon and lat for skyddsrum in bbox area
    for f in features:
        coords = f["geometry"]["coordinates"]
        lon_s, lat_s = coords[0][0], coords[0][1]

        # calculate distance to skyddsrum from input coordinates
        distance = haversine(lat_c, lon_c, lat_s, lon_s)
        result.append({
            "name": f["properties"].get("name", "unknown"),
            "lon": lon_s,
            "lat": lat_s,
            "distance_m": round(distance)
        })

    # return closest skyddsrum from input coordinates
    result.sort(key=lambda x: x["distance_m"])
    return result[:2]
    

In [190]:
closest_skyddsrum(lon_c=lon_c, lat_c=lat_c)

[{'name': '116291-3',
  'lon': 18.01275701,
  'lat': 59.30058993,
  'distance_m': 62},
 {'name': '170495-7',
  'lon': 18.01334922,
  'lat': 59.30053129,
  'distance_m': 87}]

## Flow of the tools
user address input (sweden_address_input()) -> get lon & lat -> closest_skyddsrum() -> get skyddsrum lon & lat -> 
function sweden API (sweden_skyddsrum_address()) -> get address for skyddsrum